# MSM

In [15]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.special import expit, logit
import matplotlib.pyplot as plt
import seaborn as sns
import os



## Marginal Structural Models, MSM 

### 1. 방법론의 도입 이유: 시간 의존적 교란

보통의 분석 방법(예: 로지스틱 회귀)은 **시간 의존적 교란(Time-dependent confounding)** 이 존재하는 데이터에서 인과 효과를 추정할 때 편향된 결과를 낳습니다. 

MSM은 이러한 편향을 극복하고 '반사실적(Counterfactual)' 상황에서의 인과 효과를 올바르게 추정하기 위해 도입되었습니다.

### 2. 핵심 메커니즘: 피드백 루프 (Feedback Loop)

시간에 따라 처치와 공변량이 원인과 결과로 얽히는 구조를 가집니다.

#### 2.1. 피드백 루프의 특징
**중간 변수 역할($A$):** 과거의 처치가 현재의 공변량 상태를 변화시키며 처치의 효과를 결과($Y$)로 전달합니다.


**교란 변수 역할($L$):** 현재의 공변량은 다시 미래의 처치 결정에 영향을 주는 동시에 결과의 독립적인 예측 인자가 되어 효과를 왜곡합니다.

#### 2.2 일반적인 분석 적용 시의 인과적 딜레마

표준적인 회귀 분석으로는 위 두 가지 역할을 동시에 수행하는 를 적절히 처리할 수 없는 '모순'에 빠지게 됩니다.

**공변량을 통제(Adjust)할 경우:** 처치가 공변량을 개선시켜 결과에 도달하는 인과적 경로(Indirect effect)를 차단하게 됩니다. 이는 처치의 전체 효과를 과소평가하거나 왜곡하는 결과를 초래합니다.


**공변량을 통제하지 않을 경우:** 공변량과 처치 사이의 상관관계로 인해 발생하는 교란 편향(Confounding bias)을 제거할 수 없게 되어, 처치 효과가 실제보다 높거나 낮게 측정됩니다.



### 3. MSM의 해결책: 역확률 가중치 (IPTW)

MSM은 공변량을 회귀 모델에 직접 넣는 대신 가중치를 사용하여 인과적 경로를 차단하지 않으면서 교란만을 제거합니다.

#### 3.1. Robins의 안정화된 가중치 (Stabilized Weights, SW)

단순 가중치는 공변량과 처치의 상관관계가 강할 때 분산이 매우 커지는 문제가 있습니다. Robins는 이를 해결하기 위해 분자에 **처치 예측 확률**을 포함한 안정화된 가중치를 제안했습니다.
$$sw_i = \prod_{k=0}^{K} \frac{P(A_k = a_{ki} | \bar{A}_{k-1} = \bar{a}_{(k-1)i})}{P(A_k = a_{ki} | \bar{A}_{k-1} = \bar{a}_{(k-1)i}, \bar{L}_k = \bar{l}_{ki})}$$

#### 3.2. Schomaker의 이중 로버스트 TMLE
Schomaker (2023)은 IPTW의 한계를 극복하기 위해 **이중 로버스트(Doubly Robust)** TMLE 방식을 제안했습니다. 이 방법은 결과 모델과 성향 점수 모델 중 하나만 정확해도 일관된 추정치를 제공합니다.   

- 결과 모델 (Outcome Model, $Q$): 공변량에 따른 결과값의 기대치를 모델링합니다 ($E[Y|A, L]$)
- 처치 모델 (Treatment Model,Propensity Score $g$): 공변량에 따른 처치 확률(성향 점수)을 모델링합니다 ($P(A|L)$)
- 이점: IPTW(Robins)는 처치 모델($g$)에만 의존하지만, TMLE는 두 모델을 결합하여 인과효과값을 안정적으로 추정합니다.


##### 3.2.1 왜 TMLE인가? (Robins의 IPTW와 비교)
1. 통계적 효율성: IPTW는 성향 점수가 0이나 1에 가까워질 때 추정치가 매우 불안정해지지만, TMLE는 결과 모델을 함께 사용하여 더 낮은 분산을 가집니다.
2. 머신러닝과의 결합: TMLE는 머신러닝 알고리즘을 사용하면서도 유효한 통계적 추론(신뢰구간 도출 등)이 가능하도록 설계된 프레임워크입니다.
3. 연속형 결과 처리: Schomaker 논문에서는 연속형 결과 변수를 $[0, 1]$ 범위로 변환하여 준이항(Quasibinomial) 모델로 처리함으로써 수치적 안정성을 확보합니다.





| | Robins (2000) IPTW | Schomaker (2023) TMLE |
|---|---|---|
| 가중치 활용 | 데이터셋 전체에 가중치를 직접 곱함 | 가중치 정보를 이용해 초기 모델을 업데이트함 |
| 모델 의존성 | 처치 모델 \(g\)이 틀리면 편향 발생 | \(Q\)나 \(g\) 중 하나만 맞아도 됨 (이중 로버스트) |
| 수치적 안정성 | 가중치가 클 때 결과가 불안정함 | 상대적으로 안정적 |

In [99]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# 1. Robins (2000) Table A1 데이터 생성 (수정된 개수 반영)
# L=1: A=1(Y1:108, Y0:252), A=0(Y1:24, Y0:16)
# L=0: A=1(Y1:20, Y0:30), A=0(Y1:40, Y0:10)
data = [
    {'L0': 1, 'A0': 1, 'Y': 1, 'n': 108}, {'L0': 1, 'A0': 1, 'Y': 0, 'n': 252},
    {'L0': 1, 'A0': 0, 'Y': 1, 'n': 24},  {'L0': 1, 'A0': 0, 'Y': 0, 'n': 16},
    {'L0': 0, 'A0': 1, 'Y': 1, 'n': 20},  {'L0': 0, 'A0': 1, 'Y': 0, 'n': 30},
    {'L0': 0, 'A0': 0, 'Y': 1, 'n': 40},  {'L0': 0, 'A0': 0, 'Y': 0, 'n': 10}
]
df = pd.concat([pd.DataFrame([d]*d['n']) for d in data]).drop(columns='n').reset_index(drop=True)

# 2. IPTW 추정
ps_model = smf.logit('A0 ~ L0', data=df).fit(disp=0)
df['ps'] = ps_model.predict(df)
df['weight'] = np.where(df['A0'] == 1, 1/df['ps'], 1/(1-df['ps']))

# 가중 평균을 통한 인과 위험 차이(RD) 계산
w_y1 = (df['Y'] * df['A0'] * df['weight']).sum() / 500
w_y0 = (df['Y'] * (1 - df['A0']) * df['weight']).sum() / 500
iptw_rd = w_y1 - w_y0

print(f"IPTW Risk Difference (Paper 2): {iptw_rd:.4f}")
# 결과: -0.3200 (논문 수치와 일치)

IPTW Risk Difference (Paper 2): -0.3200


In [100]:
import statsmodels.api as sm

# 1. 결과 모델 (Q) 학습
q_model = smf.logit('Y ~ A0 + L0', data=df).fit(disp=0)
df['Q1'] = q_model.predict(df.assign(A0=1))
df['Q0'] = q_model.predict(df.assign(A0=0))
df['QA'] = q_model.predict(df)

# 2. ATT를 위한 TMLE 업데이트 (Clever Covariate 사용)
p_a1 = df['A0'].mean()
df['H_att'] = np.where(df['A0'] == 1, 0, (df['ps'] / (1 - df['ps'])) / p_a1)

# Fluctuation (업데이트)
logit_QA = np.log(df['QA'] / (1 - df['QA']))
fluc = sm.GLM(df['Y'], df['H_att'], offset=logit_QA, family=sm.families.Binomial()).fit()
eps = fluc.params[0]

# 업데이트된 카운터팩츄얼 Q0_star 계산
df['Q0_star'] = 1 / (1 + np.exp(-(np.log(df['Q0'] / (1 - df['Q0'])) + eps * (df['ps'] / (1 - df['ps'])) / p_a1)))

# ATT 계산: E[Y1|A=1] - E[Y0*|A=1]
tmle_att = df[df['A0'] == 1]['Y'].mean() - df[df['A0'] == 1]['Q0_star'].mean()
print(f"TMLE ATT (Paper 1): {tmle_att:.4f}")

TMLE ATT (Paper 1): -0.3122


/var/folders/qj/p1t8n_615hx2wgwh8350jg6h0000gn/T/ipykernel_8010/879426057.py:16: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  eps = fluc.params[0]
